# Bank Customer Value Segment — Model Training

Dataset: *Bank Customer Segmentation* (India, 1M+ transactions, 884K+ customers).

Target: **High-Value vs Low-Value** customer, defined by account balance (top 33% = High-Value), predicted using only *behavioral* features (spend, timing, activity, location) — balance itself is excluded from the feature set to avoid leakage.

Trains 5 classifiers, evaluates with Accuracy/AUC/Precision/Recall/F1/MCC, and saves all artifacts used by `app.py`.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

RANDOM_STATE = 42
SAMPLE_SIZE = 200_000
VALUE_PERCENTILE = 0.67

BASE_DIR = os.getcwd()
MODEL_DIR = os.path.join(BASE_DIR, "model")
DATA_PATH = os.path.join(BASE_DIR, "bank_transactions_raw.csv")
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Load & clean raw transaction data

In [2]:
df = pd.read_csv(DATA_PATH)
df = df.rename(columns={"TransactionAmount (INR)": "TransactionAmount"})

df = df.dropna(subset=["CustomerDOB", "CustLocation", "CustAccountBalance"])
df = df[df["TransactionAmount"] > 0]

df["CustomerDOB"] = pd.to_datetime(df["CustomerDOB"], format="%d/%m/%y", errors="coerce")
df["TransactionDate"] = pd.to_datetime(df["TransactionDate"], format="%d/%m/%y", errors="coerce")
df = df.dropna(subset=["CustomerDOB", "TransactionDate"])

future_dob = df["CustomerDOB"] > pd.Timestamp("2005-01-01")
df.loc[future_dob, "CustomerDOB"] = df.loc[future_dob, "CustomerDOB"] - pd.DateOffset(years=100)

reference_date = df["TransactionDate"].max()
df["Age"] = ((reference_date - df["CustomerDOB"]).dt.days / 365.25).astype(int)
df = df[(df["Age"] >= 15) & (df["Age"] <= 90)]
df.head()

,TransactionID,CustomerID,CustomerDOB,CustGender,CustLocation,CustAccountBalance,TransactionDate,TransactionTime,TransactionAmount,Age
0,T1,C5841053,1994-01-10,F,JAMSHEDPUR,17819.05,2016-08-02,143207,25.0,22
1,T2,C2142763,1957-04-04,M,JHAJJAR,2270.69,2016-08-02,141858,27999.0,59
2,T3,C4417068,1996-11-26,F,MUMBAI,17874.44,2016-08-02,142712,459.0,19
3,T4,C5342380,1973-09-14,F,MUMBAI,866503.21,2016-08-02,142714,2060.0,43
4,T5,C9031234,1988-03-24,F,NAVI MUMBAI,6714.43,2016-08-02,181156,1762.5,28


## 2. Define target: High-Value vs Low-Value segment

Based on account balance (top 33% = High-Value). Balance is then **excluded** from the feature set.

In [3]:
value_threshold = df["CustAccountBalance"].quantile(VALUE_PERCENTILE)
joblib.dump(value_threshold, os.path.join(MODEL_DIR, "value_threshold.pkl"))

df["ValueSegment"] = np.where(
    df["CustAccountBalance"] >= value_threshold, "High-Value", "Low-Value"
)
print("Value threshold (account balance):", round(value_threshold, 2))
df["ValueSegment"].value_counts(normalize=True)

Value threshold (account balance): 34661.24


ValueSegment
Low-Value     0.669992
High-Value    0.330008
Name: proportion, dtype: float64

## 3. Feature engineering (behavioral only — no balance)

In [4]:
df["TxnHour"] = (df["TransactionTime"] // 10000).astype(int).clip(0, 23)
df["TxnMinute"] = ((df["TransactionTime"] // 100) % 100).astype(int).clip(0, 59)
df["TxnDayOfWeek"] = df["TransactionDate"].dt.dayofweek
df["TxnDayOfMonth"] = df["TransactionDate"].dt.day
df["TxnMonth"] = df["TransactionDate"].dt.month
df["IsWeekend"] = (df["TxnDayOfWeek"] >= 5).astype(int)

df["LogTransactionAmount"] = np.log1p(df["TransactionAmount"])
df["CustomerTxnCount"] = df.groupby("CustomerID")["TransactionID"].transform("count")

location_freq = df["CustLocation"].value_counts()
df["LocationFrequency"] = df["CustLocation"].map(location_freq)

location_avg_amount = df.groupby("CustLocation")["TransactionAmount"].mean()
df["LocationAvgAmount"] = df["CustLocation"].map(location_avg_amount)
df["AmountToLocationAvgRatio"] = df["TransactionAmount"] / df["LocationAvgAmount"]

joblib.dump(location_freq, os.path.join(MODEL_DIR, "location_freq.pkl"))
joblib.dump(location_avg_amount, os.path.join(MODEL_DIR, "location_avg_amount.pkl"))

FEATURE_COLUMNS = [
    "Age", "TransactionAmount", "LogTransactionAmount", "TxnHour", "TxnMinute",
    "TxnDayOfWeek", "TxnDayOfMonth", "TxnMonth", "IsWeekend",
    "CustomerTxnCount", "LocationFrequency", "AmountToLocationAvgRatio",
]
len(FEATURE_COLUMNS)

12

## 4. Stratified sample + train/test split, then export `test_data.csv`

In [5]:
df_model = df[FEATURE_COLUMNS + ["ValueSegment"]].dropna()

if len(df_model) > SAMPLE_SIZE:
    df_model, _ = train_test_split(
        df_model, train_size=SAMPLE_SIZE, stratify=df_model["ValueSegment"],
        random_state=RANDOM_STATE
    )

print(f"Modeling on {len(df_model)} rows, {len(FEATURE_COLUMNS)} features.")

target_encoder = LabelEncoder()
y_all = target_encoder.fit_transform(df_model["ValueSegment"])
joblib.dump(target_encoder, os.path.join(MODEL_DIR, "label_encoder.pkl"))
print("Target classes:", dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_))))

X_all = df_model[FEATURE_COLUMNS]
joblib.dump(FEATURE_COLUMNS, os.path.join(MODEL_DIR, "feature_columns.pkl"))

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=RANDOM_STATE, stratify=y_all
)

test_export = X_test.copy()
test_export["ValueSegment"] = target_encoder.inverse_transform(y_test)
test_export.to_csv(os.path.join(BASE_DIR, "test_data.csv"), index=False)
print(f"Saved test_data.csv with {len(test_export)} rows.")

Modeling on 200000 rows, 12 features.
Target classes: {'High-Value': np.int64(0), 'Low-Value': np.int64(1)}


Saved test_data.csv with 40000 rows.


## 5. Scale features

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.pkl"))

['/home/claude/bank_project2/model/scaler.pkl']

## 6. Define the 5 models

Random Forest uses a modest 120 shallow trees (`max_depth=10`, `min_samples_leaf=5`) to keep the saved model file small (~5MB) while preserving accuracy.

In [7]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    "kNN": KNeighborsClassifier(n_neighbors=25, n_jobs=-1),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(
        n_estimators=120, max_depth=10, min_samples_leaf=5,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
}

## 7. Train, evaluate, and save each model

Models are saved with `joblib` compression (`compress=3`) to keep file sizes GitHub-friendly.

In [8]:
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    metrics = {
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_proba), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
    }
    results.append(metrics)
    print(metrics)

    fname = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(model, os.path.join(MODEL_DIR, fname), compress=3)

{'ML Model Name': 'Logistic Regression', 'Accuracy': 0.6938, 'AUC': 0.6912, 'Precision': 0.7084, 'Recall': 0.923, 'F1': 0.8016, 'MCC': 0.214}


{'ML Model Name': 'Decision Tree', 'Accuracy': 0.6935, 'AUC': 0.6874, 'Precision': 0.7199, 'Recall': 0.8881, 'F1': 0.7952, 'MCC': 0.2316}


{'ML Model Name': 'kNN', 'Accuracy': 0.6873, 'AUC': 0.6638, 'Precision': 0.7108, 'Recall': 0.8993, 'F1': 0.794, 'MCC': 0.2045}


{'ML Model Name': 'Naive Bayes', 'Accuracy': 0.686, 'AUC': 0.6822, 'Precision': 0.6976, 'Recall': 0.938, 'F1': 0.8001, 'MCC': 0.1768}


{'ML Model Name': 'Random Forest', 'Accuracy': 0.6992, 'AUC': 0.7034, 'Precision': 0.7146, 'Recall': 0.9174, 'F1': 0.8034, 'MCC': 0.2354}


## 8. Save metrics (JSON + CSV) and display comparison table

In [9]:
with open(os.path.join(MODEL_DIR, "metrics.json"), "w") as f:
    json.dump(results, f, indent=2)

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(MODEL_DIR, "metrics_table.csv"), index=False)
results_df

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.6938,0.6912,0.7084,0.9230,0.8016,0.2140
1,Decision Tree,0.6935,0.6874,0.7199,0.8881,0.7952,0.2316
2,kNN,0.6873,0.6638,0.7108,0.8993,0.7940,0.2045
3,Naive Bayes,0.6860,0.6822,0.6976,0.9380,0.8001,0.1768
4,Random Forest,0.6992,0.7034,0.7146,0.9174,0.8034,0.2354
